# 🧠 Transformer LITE: fNIRS Mental Workload Classification

**Author:** HM Jemima  
**Dataset:** Tufts fNIRS2MW  
**Version:** LITE — Optimized for Kaggle Free Tier (~2–3 hours)  

### What changed from the full version?
| Setting | Full | Lite | Why |
|---------|------|------|-----|
| Max epochs | 100 | 50 | Most folds converge by epoch 30–40 |
| Early stopping patience | 20 | 10 | Stops sooner once plateaued |
| Batch size | 32 | 64 | Fewer gradient steps per epoch |
| Embed dim | 128 | 64 | Smaller model, same architecture |
| FF dim | 256 | 128 | Proportionally reduced |
| n_blocks | 2 | 2 | Same depth (kept for fair comparison) |
| Final deployment model | Yes | No | Saves ~30 min |

**Architecture is IDENTICAL** — same Transformer encoder. Only sizes reduced.

---

In [ ]:
#@title 1. Setup & Imports

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import warnings
import glob
import json
import pickle
import joblib
from datetime import datetime
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("="*60)
print("Transformer LITE - Setup Complete")
print("="*60)
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Time: {datetime.now().strftime('%H:%M:%S')}")
print("="*60)

In [ ]:
#@title 2. Configuration (LITE)

WINDOW_SIZE = '30sec_150ts'

BASE_PATH = '/kaggle/input/datasets/hmjemima/tufts-fnirs2mw/slide_window_data/'
OUTPUT_PATH = '/kaggle/working/Transformer_Results/'

CONFIG = {
    'data_path': f'{BASE_PATH}size_{WINDOW_SIZE}_stride_03ts/',
    'output_path': OUTPUT_PATH,
    'window_size_key': WINDOW_SIZE,
    
    'quick_test': False,
    'max_subjects': None,
    
    'n_channels': 8,
    'n_classes': 4,
    'timesteps': 150,
    
    'feature_cols': ['AB_I_O', 'AB_PHI_O', 'AB_I_DO', 'AB_PHI_DO', 
                     'CD_I_O', 'CD_PHI_O', 'CD_I_DO', 'CD_PHI_DO'],
    
    # === LITE TRAINING SETTINGS ===
    'batch_size': 64,
    'max_epochs': 50,
    'patience': 10,
    'learning_rate': 0.001,
    
    # === LITE MODEL SETTINGS ===
    'embed_dim': 64,       # was 128
    'n_heads': 4,
    'n_blocks': 2,         # kept same for fair comparison
    'ff_dim': 128,         # was 256
    'dense_units': 48,     # was 64
    'dropout': 0.4,
}

for d in ['', 'figures']:
    os.makedirs(os.path.join(OUTPUT_PATH, d), exist_ok=True)

print("\u2705 LITE Config loaded")
print(f"\ud83d\udcc2 Data: {CONFIG['data_path']}")

In [ ]:
#@title 3. Verify Data Path

if os.path.exists(CONFIG['data_path']):
    csv_files = glob.glob(os.path.join(CONFIG['data_path'], "*.csv"))
    print(f"\u2705 Found {len(csv_files)} CSV files")
else:
    print("\u274c Path not found. Searching...")
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.csv') for f in files):
            print(f"   Found CSVs in: {root}")

In [ ]:
#@title 4. Load Dataset

def load_dataset(config):
    all_X, all_y, all_subjects = [], [], []
    csv_files = sorted(glob.glob(os.path.join(config['data_path'], "*.csv")))
    if config.get('max_subjects'):
        csv_files = csv_files[:config['max_subjects']]
    print(f"Loading {len(csv_files)} subjects...")
    for idx, filepath in enumerate(tqdm(csv_files)):
        subject_id = idx + 1
        df = pd.read_csv(filepath)
        for chunk_id in df['chunk'].unique():
            chunk = df[df['chunk'] == chunk_id]
            X = chunk[config['feature_cols']].values.astype(np.float32)
            label = int(chunk['label'].iloc[0])
            if len(X) >= config['timesteps']:
                X = X[:config['timesteps']]
            else:
                X = np.pad(X, ((0, config['timesteps'] - len(X)), (0, 0)), mode='edge')
            all_X.append(X)
            all_y.append(label)
            all_subjects.append(subject_id)
    X = np.array(all_X, dtype=np.float32)
    y = np.array(all_y, dtype=np.int32)
    subjects = np.array(all_subjects, dtype=np.int32)
    X = np.nan_to_num(X)
    return X, y, subjects

X, y, subjects = load_dataset(CONFIG)
print(f"\n\u2705 Data loaded! X:{X.shape} | Subjects:{len(np.unique(subjects))} | Classes:{np.bincount(y)}")

In [ ]:
#@title 5. Transformer Model (same architecture, smaller dims)

def build_transformer(input_shape, n_classes, config):
    inputs = layers.Input(shape=input_shape)
    
    # Embedding
    x = layers.Conv1D(config['embed_dim'], 5, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Transformer Blocks
    for _ in range(config['n_blocks']):
        attn = layers.MultiHeadAttention(
            num_heads=config['n_heads'],
            key_dim=config['embed_dim'] // config['n_heads'],
            dropout=0.1
        )(x, x)
        x = layers.LayerNormalization()(x + attn)
        ffn = layers.Dense(config['ff_dim'], activation='relu')(x)
        ffn = layers.Dropout(config['dropout'])(ffn)
        ffn = layers.Dense(config['embed_dim'])(ffn)
        x = layers.LayerNormalization()(x + ffn)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(config['dense_units'], activation='relu')(x)
    x = layers.Dropout(config['dropout'])(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='Transformer')

test_model = build_transformer((CONFIG['timesteps'], CONFIG['n_channels']), CONFIG['n_classes'], CONFIG)
print(f"\u2705 Transformer LITE: {test_model.count_params():,} parameters")
del test_model
K.clear_session()

In [ ]:
#@title 6. Progress Callback

class ProgressCallback(Callback):
    def __init__(self, fold_num, total_folds):
        super().__init__()
        self.fold_num = fold_num
        self.total_folds = total_folds
        self.start_time = None
    def on_train_begin(self, logs=None):
        self.start_time = datetime.now()
        print(f"      Training started at {self.start_time.strftime('%H:%M:%S')}")
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0 or epoch == 0:
            loss = logs.get('loss', 0)
            acc = logs.get('accuracy', 0) * 100
            val_loss = logs.get('val_loss', 0)
            val_acc = logs.get('val_accuracy', 0) * 100
            print(f"      Epoch {epoch+1:3d}: loss={loss:.4f}, acc={acc:.1f}%, val_loss={val_loss:.4f}, val_acc={val_acc:.1f}%")
    def on_train_end(self, logs=None):
        duration = datetime.now() - self.start_time
        print(f"      Training completed in {duration.total_seconds():.1f}s")

print("\u2705 Progress callback defined")

In [ ]:
#@title 7. LOSO Training

def train_loso(X, y, subjects, model_fn, config):
    logo = LeaveOneGroupOut()
    unique_subj = np.unique(subjects)
    n_subjects = len(unique_subj)
    all_y_true, all_y_pred, subject_results = [], [], []
    
    print("\n" + "="*70)
    print(f"LOSO CROSS-VALIDATION | {n_subjects} subjects")
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    
    fold = 0
    for train_idx, test_idx in logo.split(X, y, subjects):
        fold += 1
        test_subj = subjects[test_idx[0]]
        print(f"\n{'\u2500'*70}")
        print(f"\ud83d\udcca FOLD {fold}/{n_subjects} | Test Subject: {test_subj}")
        print(f"   Train: {len(train_idx)} | Test: {len(test_idx)}")
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_norm = scaler.fit_transform(X_train.reshape(len(X_train), -1)).reshape(X_train.shape)
        X_test_norm = scaler.transform(X_test.reshape(len(X_test), -1)).reshape(X_test.shape)
        
        model = model_fn((X_train.shape[1], X_train.shape[2]), config['n_classes'], config)
        model.compile(optimizer=keras.optimizers.Adam(config['learning_rate']),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        callbacks = [
            ProgressCallback(fold, n_subjects),
            EarlyStopping(monitor='val_loss', patience=config['patience'],
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                              min_lr=1e-6, verbose=1)
        ]
        
        model.fit(X_train_norm, y_train, validation_split=0.15,
                  batch_size=config['batch_size'], epochs=config['max_epochs'],
                  callbacks=callbacks, verbose=0)
        
        y_pred = np.argmax(model.predict(X_test_norm, verbose=0), axis=1)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        print(f"   \u2705 FOLD {fold}: Acc={acc*100:.2f}% | F1={f1*100:.2f}%")
        
        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)
        subject_results.append({'subject': int(test_subj), 'accuracy': acc, 'f1': f1})
        K.clear_session()
        del model
        print(f"   \ud83d\udcc8 Running Avg: {np.mean([r['accuracy'] for r in subject_results])*100:.2f}%")
    
    all_y_true, all_y_pred = np.array(all_y_true), np.array(all_y_pred)
    overall_acc = accuracy_score(all_y_true, all_y_pred)
    overall_f1 = f1_score(all_y_true, all_y_pred, average='weighted')
    overall_kappa = cohen_kappa_score(all_y_true, all_y_pred)
    subj_accs = [r['accuracy'] for r in subject_results]
    mean_acc, std_acc = np.mean(subj_accs), np.std(subj_accs)
    ci_95 = 1.96 * std_acc / np.sqrt(len(subj_accs))
    
    print("\n" + "="*70)
    print(f"\ud83c\udf89 FINAL: Acc={overall_acc*100:.2f}% | F1={overall_f1*100:.2f}% | Kappa={overall_kappa:.4f}")
    print(f"   Mean\u00b1Std: {mean_acc*100:.2f}%\u00b1{std_acc*100:.2f}% | 95%CI: [{(mean_acc-ci_95)*100:.2f}%, {(mean_acc+ci_95)*100:.2f}%]")
    print("="*70)
    
    return {'overall_accuracy': overall_acc, 'overall_f1': overall_f1,
            'overall_kappa': overall_kappa, 'mean_accuracy': mean_acc,
            'std_accuracy': std_acc, 'ci_95': ci_95,
            'subject_results': subject_results,
            'confusion_matrix': confusion_matrix(all_y_true, all_y_pred).tolist(),
            'y_true': all_y_true.tolist(), 'y_pred': all_y_pred.tolist()}

print("\u2705 LOSO function defined")

In [ ]:
#@title 8. 🚀 RUN TRAINING

print("#"*70)
print("# STARTING TRANSFORMER LITE TRAINING")
print("# Expected time: ~2-3 hours on T4 GPU")
print("#"*70)

results = train_loso(X, y, subjects, build_transformer, CONFIG)

In [ ]:
#@title 9. Save Results

pkl_path = os.path.join(OUTPUT_PATH, f'results_{WINDOW_SIZE}.pkl')
save_data = {'experiment': 'Transformer', 'author': 'HM Jemima', 'version': 'LITE',
             'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
             'config': CONFIG, 'results': results}
with open(pkl_path, 'wb') as f:
    pickle.dump(save_data, f)
print(f"\u2705 Saved: {pkl_path}")

json_path = os.path.join(OUTPUT_PATH, f'summary_{WINDOW_SIZE}.json')
summary = {'accuracy': f"{results['overall_accuracy']*100:.2f}%",
           'f1': f"{results['overall_f1']*100:.2f}%",
           'kappa': f"{results['overall_kappa']:.4f}",
           'mean_std': f"{results['mean_accuracy']*100:.2f}% \u00b1 {results['std_accuracy']*100:.2f}%"}
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\u2705 Saved: {json_path}")

In [ ]:
#@title 10. Visualizations

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
cm = np.array(results['confusion_matrix'])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', ax=ax1,
            xticklabels=['0-back','1-back','2-back','3-back'],
            yticklabels=['0-back','1-back','2-back','3-back'])
ax1.set_title(f'Transformer Confusion Matrix\nAccuracy: {results["overall_accuracy"]*100:.2f}%')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')

ax2 = axes[1]
subj_accs = [r['accuracy']*100 for r in results['subject_results']]
ax2.bar(range(1, len(subj_accs)+1), subj_accs, color='forestgreen', alpha=0.7)
ax2.axhline(72.2, color='blue', linestyle='--', linewidth=2, label='Tufts RF (72.2%)')
ax2.axhline(results['overall_accuracy']*100, color='red', linestyle='-', linewidth=2,
            label=f'Transformer ({results["overall_accuracy"]*100:.1f}%)')
ax2.set_xlabel('Subject'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Per-Subject Accuracy'); ax2.legend(); ax2.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'figures', 'results.png'), dpi=300)
plt.show()

In [ ]:
#@title 11. Final Summary

print("\n" + "="*70)
print(f"\ud83c\udf89 Transformer LITE: Acc={results['overall_accuracy']*100:.2f}% | F1={results['overall_f1']*100:.2f}% | Kappa={results['overall_kappa']:.4f}")
print(f"   Tufts RF: 72.20% | Improvement: {(results['overall_accuracy']*100 - 72.2):+.2f}%")
print("="*70)

In [ ]:
#@title 12. Download Results
import shutil
shutil.make_archive('/kaggle/working/Transformer_Results', 'zip', OUTPUT_PATH)
print("\u2705 Created: /kaggle/working/Transformer_Results.zip")